# Lecture 3: QLoRA Fine-Tuning LLaMA-3 — Domain Adaptation for Digital Humanities Terminology

In Lecture 1 we loaded LLaMA-3 and used it to auto-summarise short-video transcripts. In Lecture 2 we used CLIP cross-modal encodings to align video frames with classical-text passages.
However, a general-purpose pretrained model is often vague — or actively wrong — when asked to explain specialised Digital Humanities terms such as *kè běn* (woodblock edition), *shàn xiě* (manuscript copying), or *jīn shí xué* (epigraphy).

**Goal for this lecture**: use **QLoRA** (Quantised Low-Rank Adaptation) to fine-tune LLaMA-3 so that it produces accurate, reproducible explanations of DH terminology when given short-video transcript excerpts as input — a foundational step toward a domain-specific knowledge assistant.

# Lecture Structure

## 1  Environment setup and GPU check
## 2  Loading LLaMA-3 in 4-bit quantisation (BitsAndBytes NF4)
## 3  Checkpoint 1: confirm the base model runs before touching LoRA
## 4  LoRA configuration for parameter-efficient fine-tuning
## 5  Building a placeholder dataset (10 examples) and tokenising
## 6  Checkpoint 2: confirm loss decreases on tiny data
## 7  How to read a loss curve
## 8  Scaling to the full corpus (parameterised template)
## 9  Saving the adapter (reused in Lecture 4)
## Quick Check

# Transparency Note

> **Model source**: this lecture uses `NousResearch/Meta-Llama-3-8B-Instruct`, a **community-hosted mirror** of the official Meta Llama 3 weights on Hugging Face. No Hugging Face account or access token is required. The original model is released under the Meta Llama 3 Community License Agreement.

# Chapter 1  Environment Setup and GPU Check

- Install four key libraries: `transformers` (model), `peft` (LoRA), `bitsandbytes` (4-bit quantisation), `datasets` (dataset handling)
- Run a GPU check — if no GPU is detected, the script raises a clear error immediately rather than silently falling back to CPU

> **Local users (mllm-dh environment)**: if you are already running inside the `mllm-dh` conda environment, you can skip the `!pip install` line and run only the GPU check.

In [ ]:
# ===== Chapter 1: Environment Setup =====
# Colab users: run this cell to install dependencies automatically.
# Local mllm-dh environment users: comment out the !pip line and run only the GPU check.

#!pip install -q -U transformers accelerate peft bitsandbytes datasets

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. In Colab go to: Runtime → Change runtime type → T4 GPU.'
        ' The 4-bit quantised model cannot run effectively on CPU.'
    )

device_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU ready: {device_name}')
print(f'Total VRAM: {total_mem:.1f} GB')

# Chapter 2  Loading LLaMA-3 in 4-bit Quantisation

## Why quantise?

LLaMA-3 8B at full precision (fp32) requires roughly **32 GB** of VRAM — far beyond what the Colab free tier provides.
**4-bit NF4 quantisation** compresses each weight from 32 bits to 4 bits, reducing the VRAM footprint to around **5–6 GB**, while double quantisation further compresses the quantisation constants themselves.

> Intuition: quantisation is lossy compression, but for the vast majority of inference and fine-tuning tasks the accuracy loss is negligible in practice.

In [ ]:
# ===== Chapter 2: Load Tokeniser and Quantisation Config =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'NousResearch/Meta-Llama-3-8B-Instruct'

# Pick compute dtype based on GPU capability (T4 and older → fp16; Ampere and newer → bf16)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Compute dtype: {compute_dtype}')

# 4-bit NF4 quantisation config with double quantisation
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# Llama-3 has no pad_token by default — we must set one explicitly
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Tokeniser loaded.')

In [ ]:
# Load the 4-bit quantised model
# First run downloads ~5 GB of weights — allow 3–5 minutes on Colab
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.use_cache = False  # disable KV cache during fine-tuning to save VRAM

print('Model loaded.')
print(f'Parameter count: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B')

# Chapter 3  Checkpoint 1: Confirm Base Model Runs

Before touching any LoRA configuration, run a single inference call to verify the model loaded correctly.

This is a **deliberate checkpoint**: if the base model has a loading problem (corrupted weights, incompatible quantisation parameters, etc.) it will fail loudly here, before you invest time in LoRA setup and training.

In [ ]:
# ===== Checkpoint 1: confirm base model works before proceeding =====

test_prompt = 'In one sentence, what is Digital Humanities?'
messages = [{'role': 'user', 'content': test_prompt}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(formatted, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== Checkpoint 1 output ===')
print(response)
print('=' * 26)
print('Base model is working — proceed to LoRA setup.')

# Chapter 4  LoRA: The Intuition Behind Parameter-Efficient Fine-Tuning

## Why not full fine-tuning?

Full fine-tuning requires storing gradients and optimiser states for all 8 billion parameters — well over 60 GB of VRAM.

**LoRA's idea**: freeze the original model weights and insert two small low-rank matrices (A and B, where rank = r) alongside the key linear layers in each attention block. During training only these small matrices are updated; at inference their product is added back to the original weights.

The result: trainable parameters are typically **less than 1%** of the original model, yet enough to adapt the model's vocabulary and style to a new domain.

In [ ]:
# ===== Chapter 4: LoRA Configuration =====
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for 4-bit quantised training (enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                       # rank of the low-rank matrices (higher = more capacity, more VRAM)
    lora_alpha=32,              # scaling factor, conventionally set to 2×r
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,          # light dropout to reduce overfitting
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,}')
print(f'Total parameters:     {total:,}')
print(f'Trainable fraction:   {100 * trainable / total:.4f}%')

# Chapter 5  Building the Placeholder Dataset

## Data format: instruction following

Each training example has three parts, following Llama-3's chat template:

| Role | Content |
|------|---------|
| system | Establishes the model as a DH domain expert |
| user | Video transcript excerpt containing the term to explain |
| assistant | Accurate professional explanation (training target) |

The 10 examples below are **placeholder samples** covering core DH vocabulary (textual criticism, archival practice, epigraphy, etc.). Replace them with your own video transcript data before running a real training run.

In [ ]:
# ===== Chapter 5: Placeholder Dataset (10 examples) =====
# ⚠️  These are placeholder examples — replace with real transcript data before training.

SYSTEM_PROMPT = (
    'You are a scholar specialising in Digital Humanities. '
    'You explain terminology from classical texts, cultural heritage, and archival practice accurately and concisely.'
)

RAW_DATA = [
    {
        'transcript': 'The video says this book is a ke ben, meaning it is different from a handwritten copy.',
        'explanation': 'Ke ben (刻本, woodblock edition) refers to a book produced by woodblock printing: a craftsman carved the text in reverse onto a wooden block, applied ink, and pressed it onto paper. Song and Yuan dynasty woodblock editions are primary artefacts for textual studies.',
    },
    {
        'transcript': 'The term shan xie appears repeatedly in the document. What does it mean?',
        'explanation': 'Shan xie (缮写, manuscript copying) refers to the professional scribal process of transcribing and correcting a text. Before the spread of printing it was the primary means of textual transmission and provides key evidence for the history of script forms.',
    },
    {
        'transcript': 'The researcher says his specialisation is jin shi xue.',
        'explanation': 'Jin shi xue (金石学, epigraphy) is the traditional Chinese study of bronze inscriptions (jin) and stone carvings (shi). It is the predecessor of modern Chinese epigraphy and archaeology, providing primary sources for historical geography and script evolution.',
    },
    {
        'transcript': 'When arranging the archives they had to follow the principle of original order.',
        'explanation': 'The Principle of Original Order is a core archival principle: records should be preserved in the sequence established by their creator so that contextual relationships and evidential value are maintained.',
    },
    {
        'transcript': 'After digitisation the team annotated every image.',
        'explanation': 'Digital annotation refers to attaching structured metadata (such as personal names, place names, dynasties, or subject headings) to digitised images or text, enabling computational retrieval and humanities analysis.',
    },
    {
        'transcript': 'There are variant readings across different versions, so the editors conducted jiao kan.',
        'explanation': 'Jiao kan (校勘, textual criticism) is a core philological method: by comparing different textual witnesses of the same work, scholars identify and evaluate variant readings (yi wen) with the goal of recovering the most authentic text.',
    },
    {
        'transcript': 'This is a transmitted text, which differs from excavated documents.',
        'explanation': 'A transmitted text (传世本, chuan shi ben) is a version of a work that has been copied and printed continuously down to the present; an excavated document is a physical text recovered through archaeology (such as bamboo slips or silk manuscripts). Cross-checking the two is a key philological method.',
    },
    {
        'transcript': 'The museum displayed several rubbings from stone stelae.',
        'explanation': 'A rubbing (拓片, ta pian) is a flat reproduction made by laying paper over an inscribed stone or bronze surface and pressing ink-soaked cloth against it. Rubbings are the primary medium for transmitting epigraphic texts and are a priority target for digital capture.',
    },
    {
        'transcript': 'His doctoral thesis is on mu lu xue.',
        'explanation': 'Mu lu xue (目录学, bibliography) is the study of methods for recording, classifying, and cataloguing documents. It holds an independent disciplinary position in the Chinese scholarly tradition; modern digital-library metadata standards such as MARC and Dublin Core can be seen as its digital extension.',
    },
    {
        'transcript': 'The video introduces a digital twin project for a cultural heritage site.',
        'explanation': 'A digital twin in the cultural heritage context refers to a high-precision digital model of a site or artefact created through 3D scanning, photogrammetry, or similar techniques. It is used for remote study, virtual exhibition, and damage prevention.',
    },
]

print(f'Dataset size: {len(RAW_DATA)} examples')
print('Example (first entry):')
print('  Transcript:', RAW_DATA[0]['transcript'])
print('  Explanation (50 chars):', RAW_DATA[0]['explanation'][:50], '...')

In [ ]:
# ===== Chapter 5 (cont.): Tokenisation Function =====
from datasets import Dataset
from transformers import default_data_collator

MAX_LENGTH = 512

def format_and_tokenize(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': example['transcript']},
        {'role': 'assistant', 'content': example['explanation']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokenized = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
    )
    # Set padding positions to -100 so they are excluded from the loss calculation
    tokenized['labels'] = [
        -100 if mask == 0 else iid
        for iid, mask in zip(tokenized['input_ids'], tokenized['attention_mask'])
    ]
    return tokenized

hf_dataset = Dataset.from_list(RAW_DATA)
tokenized_dataset = hf_dataset.map(
    format_and_tokenize,
    remove_columns=['transcript', 'explanation'],
)

print(f'Tokenisation complete. Examples: {len(tokenized_dataset)}')
print(f'Sequence length: {len(tokenized_dataset[0]["input_ids"])}')

# Chapter 6  Checkpoint 2: Confirm Loss Decreases on Tiny Data

## Why train on 10 examples first?

Training on 10 samples is not meant to teach the model anything useful — it is a **pipeline sanity check** to confirm that data flows correctly through forward pass → backward pass → parameter update.

**Expected result**: loss drops from roughly 2–3 to below 1 within 3 epochs. If it does not, see the next chapter for diagnosis.

| Hyperparameter | Value | Reason |
|----------------|-------|--------|
| batch size | 1 | low-VRAM environment |
| gradient accumulation | 4 | effective batch size = 4 |
| epochs | 3 | pipeline validation only |
| learning rate | 2e-4 | standard LoRA range |

In [ ]:
# ===== Checkpoint 2: confirm loss decreases on tiny data before scaling up =====
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./qlora_checkpoint',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_strategy='no',
    report_to='none',               # no wandb or other external logging integrations
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=default_data_collator,
)

print('Starting training (Checkpoint 2: tiny-data validation)...')
trainer.train()
print('Training complete.')

In [ ]:
# Print per-step loss history
loss_history = [
    (entry['step'], entry['loss'])
    for entry in trainer.state.log_history
    if 'loss' in entry
]

print(f'{"Step":>6}  {"Loss":>10}')
print('-' * 20)
for step, loss in loss_history:
    print(f'{step:>6}  {loss:>10.4f}')

# Chapter 7  How to Read the Loss Curve

## A healthy curve (10 examples, 3 epochs)

- Initial loss around **2–3** (the model is essentially guessing)
- Loss decreases steadily; ends **below 1**, ideally below 0.5
- Fluctuations are fine as long as the overall direction is downward

## Warning signs and most likely causes

| Symptom | Most likely cause |
|---------|------------------|
| Loss completely flat | `target_modules` is wrong so LoRA was never injected; or learning rate is too low (< 1e-5) |
| Loss becomes NaN | Learning rate too high (> 5e-3); bf16/fp16 numerical overflow |
| Loss explodes (rises rapidly) | Gradient clipping not enabled; batch size too small causing noisy gradients |
| Loss extremely low (< 0.01) but inference is incoherent | Overfitting to 10 examples; label/padding masking bug causing the model to memorise answers |

> Reminder: the purpose of training on 10 examples is to **validate the pipeline**, not to minimise loss.

# Chapter 8  Scaling to the Full Corpus

## From pipeline validation to real training

Once Checkpoint 2 passes, only two things need to change:
1. **Data source**: load from a JSON/CSV file instead of a hardcoded list
2. **Epoch count**: the larger the corpus, the fewer epochs needed to avoid overfitting

**Colab free-tier time budget (rough guide):**

| Examples | Suggested epochs | Estimated time (T4) |
|----------|-----------------|---------------------|
| ≤ 500 | 3–5 | < 1 hour |
| 500–2 000 | 1–2 | 1–3 hours |
| > 2 000 | 1 + save checkpoints | > 3 hours (may need session resumption) |

In [ ]:
# ===== Chapter 8: Full-Corpus Training Template =====
# ⚠️  Run this cell only after Checkpoint 2 has passed (loss decreasing normally).
# Edit only the configuration section below — the rest of the code stays the same.

import json
from datasets import Dataset

# --- Configuration (edit only this section) ---
DATA_PATH = 'path/to/your_data.json'   # each record must have 'transcript' and 'explanation' keys
NUM_EPOCHS = 2                          # reduce for larger corpora
SAVE_PATH = './qlora_adapter_full'

# --- Load real data (uncomment to replace placeholder dataset) ---
# with open(DATA_PATH, 'r', encoding='utf-8') as f:
#     full_raw_data = json.load(f)

# Placeholder: still using the small dataset for demonstration
full_raw_data = RAW_DATA

full_dataset = Dataset.from_list(full_raw_data)
full_tokenized = full_dataset.map(
    format_and_tokenize,
    remove_columns=['transcript', 'explanation'],
)
print(f'Full dataset size: {len(full_tokenized)} examples')

full_training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',              # save checkpoints per epoch for full training runs
    report_to='none',
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    remove_unused_columns=False,
)

full_trainer = Trainer(
    model=model,
    args=full_training_args,
    train_dataset=full_tokenized,
    data_collator=default_data_collator,
)

# Uncomment to run full training:
# full_trainer.train()
# print('Full training complete.')
print('(Full training is commented out — uncomment and replace with real data to run.)')

# Chapter 9  Saving the LoRA Adapter

The output of LoRA fine-tuning is not a full 8B-parameter model, but a set of **incremental weights** (typically tens of MB) that sit alongside the frozen base model.

To use the adapter later, load it on top of the same base model — no need to re-download the full weights.

> **Lecture 4 preview**: the saved adapter will be reloaded in the next lecture to build an automated DH terminology processing pipeline over the full video corpus.

In [ ]:
# ===== Chapter 9: Save LoRA Adapter and Tokeniser =====
import os

ADAPTER_SAVE_PATH = './qlora_adapter_lecture3'

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f'Adapter saved to: {ADAPTER_SAVE_PATH}')
print('Saved files:')
for fname in sorted(os.listdir(ADAPTER_SAVE_PATH)):
    fsize = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, fname))
    print(f'  {fname:<40}  {fsize / 1e6:.2f} MB')

# Quick Check

**What you should take away from this lecture:**

- **QLoRA loading**: 4-bit NF4 quantisation compresses LLaMA-3 8B to ~5–6 GB VRAM, making it usable on a Colab free-tier T4 GPU
- **LoRA injection**: by freezing the base model and training only low-rank matrices in the attention layers (< 1% of parameters), we adapt domain vocabulary while preserving the model's general capabilities
- **Two-checkpoint strategy**: verify the base model can infer (Checkpoint 1), then verify training reduces loss (Checkpoint 2) — this catches pipeline errors before wasting compute on a full training run
- **Reusable template**: the Chapter 8 code block is parameterised to accept any JSON/CSV transcript dataset; the saved adapter from Chapter 9 is the starting point for Lecture 4's corpus-processing pipeline

> Continuity across lectures: L01 made transcripts short and readable; L02 aligned visual evidence with classical text passages; **L03 gives the model genuine domain knowledge of DH terminology.**